# Industrial Load Flexibility at a Steel Plant: shed, shift or shave?

**Question:** How much do the electricity bill and the peak demand change if a steel plant sheds, shifts or shaves load, and how much of that depends on the rate design?

## Data source and reference
- **Source:** https://archive.ics.uci.edu/dataset/851/steel+industry+energy+consumption
- **Reference:** VE, S., Shin, C., & Cho, Y. (2021). Efficient energy consumption prediction model for a data analytic-enabled industry building in a smart city. *Building Research & Information, 49*(1), 127-143.
- The file `Steel_industry_data.csv` has 15-minute readings for one steel plant for all of 2018.

## Key terms
| Term | Meaning |
|---|---|
| **kW** | Power: How much power is the plant is drawing right now. |
| **kWh** | Energy: kW multiplied by hours. |
| **Energy charge** | $/kWh. On a time-of-use (TOU) tariff the price depends on the hour. |
| **Demand charge** | $/kW-month, applied to the highest 15-minute kW reading in the month. One short spike sets the charge for the whole month. |
| **Peak window** | Weekdays 4-9pm here, when the grid is stressed and TOU prices are highest. |

## Three levers for flexibility
| Lever | What the plant does | Example |
|---|---|---|
| **Shed** | Cuts load in the peak window and does not make it up. Output is lost. | Pause a non-critical process. |
| **Shift** | Moves load out of the peak window to other hours of the same day. | Run a batch earlier or later. |
| **Shave** | Caps the monthly peak kW at any hour and moves the clipped energy to quieter hours. | Stagger large motors so they don't start together. |

## Assumptions you can change
- The plant is scaled by `SCALE` to represent a larger facility.
- The two tariffs are **illustrative rate designs, not real utility tariffs**. Replace them with URDB values for a specific utility.
- The flex share `FRAC` is an assumption, not a process limit.

## 1. Load the data and look at a normal day

**What to look for**
- *Left:* the average weekday and weekend shape by hour. The orange band is the peak window.
- *Right:* the highest 15-minute load in each month, with the hour it happened. **Orange bars** peak inside the window and **blue bars** peak outside it. A blue month cannot get a lower demand charge just by cutting load in the window.

In [ ]:
# Source: https://archive.ics.uci.edu/dataset/851/steel+industry+energy+consumption
# Reference: VE, S., Shin, C., & Cho, Y. (2021). Efficient energy consumption prediction model for a data
# analytic-enabled industry building in a smart city. Building Research & Information, 49(1), 127-143.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

plt.rcParams.update({'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})

CSV = 'Steel_industry_data.csv'   # same folder as this notebook
SCALE = 10                        # specific dataset tracks a smart, small-scale steel processing facility, not a primary smelting plant.• Because it does not run an Electric Arc Furnace to melt scrap metal down from scratch, its power draw drops from the hundreds of megawatts down to the hundreds of kilowatts.

PEAK_START, PEAK_END = 16, 21     # weekday peak window, hours [start, end)

raw = pd.read_csv(CSV)
ts_end = pd.to_datetime(raw['date'], format='%d/%m/%Y %H:%M')
assert (ts_end.dt.day_name() == raw['Day_of_week']).mean() > 0.99, 'dates are not day-first'

# Timestamps are interval-ending, and 00:00 is the last interval of its own date (24:00).
# Roll those to the next day, then shift to interval-start so every row falls in 2018.
is_midnight = (ts_end.dt.hour == 0) & (ts_end.dt.minute == 0)
idx = ts_end.where(~is_midnight, ts_end + pd.Timedelta(days=1)) - pd.Timedelta(minutes=15)
load = pd.Series(raw['Usage_kWh'].to_numpy() * 4 * SCALE, index=idx, name='kW')  # kWh per 15 min -> kW
assert load.index.is_unique and load.index.is_monotonic_increasing

cal = pd.DataFrame(index=load.index)
cal['month'] = load.index.month
cal['summer'] = cal['month'].between(6, 9)
cal['weekday'] = load.index.dayofweek < 5
cal['peak'] = cal['weekday'] & (load.index.hour >= PEAK_START) & (load.index.hour < PEAK_END)
cal['date'] = load.index.normalize()

print(f'{len(load):,} intervals, {load.index.min()} to {load.index.max()}')
print(f'average {load.mean()/1000:.2f} MW, peak {load.max()/1000:.2f} MW, annual {load.sum()*0.25/1e6:.1f} GWh')

# Plot 1: a typical day, and when each month's peak happens
hourly = load.groupby([cal['weekday'], load.index.hour]).mean() / 1000
mp = load.groupby(cal['month']).max() / 1000
when = load.groupby(cal['month']).idxmax()
in_win = cal.loc[when.to_numpy(), 'peak'].to_numpy()
print(f'monthly peaks inside the peak window: {int(in_win.sum())} of 12')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for is_wd, label, color in ((True, 'Weekday', 'tab:blue'), (False, 'Weekend', 'tab:green')):
    ax1.plot(hourly[is_wd].index, hourly[is_wd].values, marker='o', color=color, label=label)
ax1.axvspan(PEAK_START, PEAK_END, color='orange', alpha=0.2, label='Peak window (weekdays)')
ax1.set_xlabel('Hour of day')
ax1.set_ylabel('Average load (MW)')
ax1.set_title('Plot 1a: a typical weekday and weekend')
ax1.legend()

ax2.bar(mp.index, mp.values, color=np.where(in_win, 'tab:orange', 'tab:blue'))
for m, t in zip(mp.index, when):
    ax2.text(m, mp[m] + 0.05, f'{t.hour:02d}h', ha='center', fontsize=8)
ax2.set_xticks(range(1, 13))
ax2.set_xlabel('Month')
ax2.set_ylabel('Highest 15-minute load (MW)')
ax2.set_title('Plot 1b: monthly peak (label = hour it happened)')
ax2.set_ylim(0, mp.max() * 1.25)
ax2.legend(handles=[Patch(color='tab:blue', label='Peak outside window'),
                    Patch(color='tab:orange', label='Peak inside window')], loc='upper center', ncol=2)
plt.tight_layout()
plt.show()

## 2. Testing two rate designs

- **Tariff A (TOU energy + peak demand):** The energy price jumps in the peak window, and there is an extra demand charge on the summer peak-window maximum.
- **Tariff B (flat energy + high demand charge):** Energy costs the same at every hour, but the monthly maximum is expensive.

**What to look for**
- *Left:* The price of a kWh by hour on a weekday. Only tariff A rewards moving energy out of 4-9pm.
- *Right:* The demand charges. Tariff B has only the monthly-max charge. It has no peak-window demand charge, so its second bar is $0. Tariff B puts all of its demand money on the single highest reading of the month.

In [ ]:
# Illustrative rate designs; replace with URDB values for a real utility
TARIFFS = {
    'A: TOU energy + peak demand': dict(
        energy=dict(sp=0.28, so=0.11, wp=0.15, wo=0.09),  # $/kWh: summer/winter x peak/off-peak
        demand_all=8.0,      # $/kW-month on the monthly max (all hours)
        demand_peak=14.0,    # $/kW-month on the summer peak-window max
        fixed=500.0),        # $/month
    'B: flat energy + high demand charge': dict(
        energy=dict(sp=0.055, so=0.055, wp=0.055, wo=0.055),
        demand_all=16.0, demand_peak=0.0, fixed=500.0),
}
TARIFF_COLORS = ['tab:blue', 'tab:red']

def bill(kw, t):
    e = t['energy']
    rate = np.where(cal['summer'],
                    np.where(cal['peak'], e['sp'], e['so']),
                    np.where(cal['peak'], e['wp'], e['wo']))
    energy = float((kw * 0.25 * rate).sum())
    d_all = kw.groupby(cal['month']).max().sum() * t['demand_all']
    sp = cal['summer'] & cal['peak']
    d_peak = kw[sp].groupby(cal['month'][sp]).max().sum() * t['demand_peak']
    return dict(energy=energy, demand=float(d_all + d_peak), fixed=12 * t['fixed'])

# Plot 2: what each tariff charges
hours = np.arange(24)
def weekday_price(t, summer):
    e = t['energy']
    return [e['sp' if summer else 'wp'] if PEAK_START <= h < PEAK_END else e['so' if summer else 'wo'] for h in hours]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for (name, t), color in zip(TARIFFS.items(), TARIFF_COLORS):
    for summer, style in ((True, '-'), (False, '--')):
        y = weekday_price(t, summer)
        ax1.step(np.append(hours, 24), y + [y[-1]], where='post', color=color, ls=style,
                 label=f'Tariff {name[0]} ({"summer" if summer else "winter"})')
ax1.axvspan(PEAK_START, PEAK_END, color='orange', alpha=0.15)
ax1.set_xlabel('Hour of day (weekday)')
ax1.set_ylabel('Energy price ($/kWh)')
ax1.set_title('Plot 2a: energy price by hour')
ax1.legend(fontsize=8)

w = 0.35
for i, ((name, t), color) in enumerate(zip(TARIFFS.items(), TARIFF_COLORS)):
    ax2.bar(np.arange(2) + (i - 0.5) * w, [t['demand_all'], t['demand_peak']], w, color=color, label=f'Tariff {name[0]}')
for i, ((name, t), color) in enumerate(zip(TARIFFS.items(), TARIFF_COLORS)):
    for j, val in enumerate([t['demand_all'], t['demand_peak']]):
        if val == 0:
            ax2.text(j + (i - 0.5) * w, 0.3, '$0 (none)', ha='center', color=color, fontsize=9)
ax2.set_xticks([0, 1])
ax2.set_xticklabels(['Monthly max, any hour', 'Summer peak-window max'])
ax2.set_ylabel('Demand charge ($/kW-month)')
ax2.set_title('Plot 2b: demand charges')
ax2.legend()
plt.tight_layout()
plt.show()

## 3. Applying three load flexibility levers on one day

Each panel takes one summer Tuesday and compares the baseline (gray) with the strategy (red). **Green** shading is load removed and **purple** is load added elsewhere. The orange band is the peak window.

**What to look for**
- *Shed:* green only. The energy is gone.
- *Shift:* green in the window, purple in off-peak hours. The energy is moved, not lost.
- *Shave:* the highest spikes of the day are clipped wherever they occur, and the energy reappears elsewhere.

In [ ]:
import matplotlib.dates as mdates

def shed(kw, frac):
    out = kw.copy()
    out[cal['peak']] = kw[cal['peak']] * (1 - frac)
    return out

def shift(kw, frac):
    # Headroom is capped at the baseline monthly max so the shift cannot create a new peak
    peak = cal['peak']
    out = kw.copy()
    moved_kwh = (kw[peak] * frac * 0.25).groupby(cal['date'][peak]).sum()
    out[peak] = kw[peak] * (1 - frac)
    cap = kw.groupby(cal['month']).transform('max')
    room = (cap - out).clip(lower=0).where(~peak, 0.0)
    room_kwh_day = (room * 0.25).groupby(cal['date']).sum()
    share = (moved_kwh / room_kwh_day).fillna(0).clip(upper=1)
    return out + room * cal['date'].map(share).fillna(0)

def shave(kw, frac):
    # Clip each month at (1 - frac) x its max; move clipped energy into same-day headroom under that cap
    cap = kw.groupby(cal['month']).transform('max') * (1 - frac)
    out = kw.clip(upper=cap)
    clipped_kwh = ((kw - out) * 0.25).groupby(cal['date']).sum()
    room = (cap - out).clip(lower=0)
    room_kwh_day = (room * 0.25).groupby(cal['date']).sum()
    share = (clipped_kwh / room_kwh_day).fillna(0).clip(upper=1)
    return out + room * cal['date'].map(share).fillna(0)

def evaluate(kw_new, tname):
    t = TARIFFS[tname]
    b0, b1 = bill(load, t), bill(kw_new, t)
    tot0, tot1 = sum(b0.values()), sum(b1.values())
    return dict(tariff=tname,
                energy_sav=b0['energy'] - b1['energy'],
                demand_sav=b0['demand'] - b1['demand'],
                total_sav=tot0 - tot1,
                sav_pct=100 * (tot0 - tot1) / tot0)

def peak_stats(kw):
    m = kw.groupby(cal['month']).max()
    return dict(annual_peak_kw=m.max(), avg_monthly_peak_kw=m.mean())

# Plot 3: the three levers on one summer weekday
FRAC = 0.15                       # share of load flexed (assumption)
DAY = '2018-07-10'                # a summer Tuesday
strategies = {'Shed': shed(load, FRAC), 'Shift': shift(load, FRAC), 'Peak shave': shave(load, FRAC)}
descr = {'Shed': 'cut load in the window, energy lost',
         'Shift': 'move window load to other hours',
         'Peak shave': 'clip the highest spikes, any hour'}

base_day = load[DAY] / 1000
x = base_day.index
win_start = pd.Timestamp(DAY) + pd.Timedelta(hours=PEAK_START)
win_end = pd.Timestamp(DAY) + pd.Timedelta(hours=PEAK_END)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, kw) in zip(axes, strategies.items()):
    new = kw[DAY] / 1000
    ax.plot(x, base_day.values, color='gray', lw=1.5, label='Baseline')
    ax.plot(x, new.values, color='tab:red', lw=1.5, label=name)
    ax.fill_between(x, base_day.values, new.values, where=new.values < base_day.values,
                    color='tab:green', alpha=0.5, label='Load removed')
    ax.fill_between(x, base_day.values, new.values, where=new.values > base_day.values,
                    color='tab:purple', alpha=0.5, label='Load added')
    ax.axvspan(win_start, win_end, color='orange', alpha=0.15)
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Hh'))
    ax.set_title(f'{name}: {descr[name]}', fontsize=10)
    ax.set_xlabel('Hour of day')
    ax.legend(fontsize=8, loc='upper left')
axes[0].set_ylabel('Load (MW)')
fig.suptitle(f'Plot 3: {FRAC:.0%} flex on {DAY} (orange = peak window)')
plt.tight_layout()
plt.show()

## 4. What each strategy does to the bill and the billed peak

The tables show the baseline bill, the savings for each strategy under each tariff, and the peak demand.

**What to look for**
- *Left:* how much of each baseline bill is energy versus demand. The fixed charge is only $6k a year, too thin to see. In the default run tariff B is mostly demand charge, so lowering the peak matters most there.
- *Right:* the highest 15-minute load in each month after each strategy. That is the number the demand charge is based on. The thick gray line is the baseline. Shed and shift sit on top of it because the monthly peaks happen outside 4-9pm (Plot 1b). Peak shave is lower because it caps each month at 85% of its peak.


In [ ]:
scen = {f'{k} {FRAC:.0%}': v for k, v in strategies.items()}

# Share of removed energy that found headroom (the rest is effectively shed)
moved = (load[cal['peak']] * FRAC * 0.25).sum()
recovered = (scen[f'Shift {FRAC:.0%}'].sum() - load.sum()) * 0.25 + moved
print(f'shift: {100 * recovered / moved:.1f}% of moved energy recovered off-peak')
clipped = (load - load.clip(upper=load.groupby(cal['month']).transform('max') * (1 - FRAC))).sum() * 0.25
recovered = (scen[f'Peak shave {FRAC:.0%}'].sum() - load.sum()) * 0.25 + clipped
print(f'peak shave: {100 * recovered / clipped:.1f}% of clipped energy recovered')

rows = []
for sname, kw in scen.items():
    for tname in TARIFFS:
        r = evaluate(kw, tname)
        r['scenario'] = sname
        rows.append(r)
res = pd.DataFrame(rows).set_index(['scenario', 'tariff'])

base = pd.DataFrame({tn: bill(load, t) for tn, t in TARIFFS.items()}).T
base['total'] = base.sum(axis=1)
peaks = pd.DataFrame({'Baseline': peak_stats(load), **{k: peak_stats(v) for k, v in scen.items()}}).T
peaks['avg_monthly_peak_change_%'] = 100 * (peaks['avg_monthly_peak_kw'] / peaks.loc['Baseline', 'avg_monthly_peak_kw'] - 1)

print('Baseline annual bill ($)')
display(base.round(0))
print('Savings vs baseline ($/yr)')
display(res.round(1))
print('Peak demand (kW)')
display(peaks.round(1))

# Plot 4: baseline bill makeup, and the billed monthly peak after each strategy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
bottom = np.zeros(len(base))
for col, color in (('energy', 'tab:blue'), ('demand', 'tab:red'), ('fixed', 'tab:gray')):
    vals = base[col].to_numpy() / 1e3
    label = f'{col.title()} charges'
    if col == 'fixed':
        label += f' (${vals[0]:,.0f}k/yr, too thin to see)'
    ax1.bar(range(len(base)), vals, bottom=bottom, color=color, label=label)
    if col != 'fixed':
        for i, v in enumerate(vals):
            ax1.text(i, bottom[i] + v / 2, f'${v:,.0f}k', ha='center', va='center', color='white')
    bottom += vals
for i, tot in enumerate(bottom):
    ax1.text(i, tot + 20, f'${tot:,.0f}k', ha='center')
ax1.set_xticks(range(len(base)))
ax1.set_xticklabels([f'Tariff {n[0]}' for n in base.index])
ax1.set_ylabel('Baseline annual bill ($ thousand)')
ax1.set_title('Plot 4a: what the baseline bill is made of')
ax1.set_ylim(0, bottom.max() * 1.4)
ax1.legend(loc='upper center', ncol=2, fontsize=8)

months = range(1, 13)
ax2.plot(months, load.groupby(cal['month']).max() / 1000, color='gray', lw=5, alpha=0.5, label='Baseline')
for (name, kw), style, color in zip(scen.items(), ('--', ':', '-'), ('tab:orange', 'tab:green', 'tab:red')):
    ax2.plot(months, kw.groupby(cal['month']).max() / 1000, style, color=color, marker='o', ms=4, label=name)
ax2.set_xticks(list(months))
ax2.set_xlabel('Month')
ax2.set_ylabel('Highest 15-minute load in the month (MW)')
ax2.set_title('Plot 4b: the monthly peak that sets the demand charge')
ax2.text(0.02, 0.03, f'{int(in_win.sum())} of 12 monthly peaks fall in the peak window, so shed and shift sit on the baseline',
         transform=ax2.transAxes, fontsize=8)
ax2.set_ylim(top=ax2.get_ylim()[1] + 0.5)
ax2.legend(loc='upper center', ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

## 5. Where the savings come from

**What to look for**
- Blue bars are savings on energy charges and red bars are savings on demand charges. The text above each group gives the total and the share of the bill.
- Compare the same strategy under tariff A and tariff B. The load change is identical, so any difference in savings comes from the rate design.

In [ ]:
# Plot 5: savings by bill component
x = np.arange(len(res))
labels = [f'{s}\nTariff {t[0]}' for s, t in res.index]
top = max(res['energy_sav'].max(), res['demand_sav'].max()) / 1e3
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - 0.2, res['energy_sav'] / 1e3, 0.4, color='tab:blue', label='Energy charges')
ax.bar(x + 0.2, res['demand_sav'] / 1e3, 0.4, color='tab:red', label='Demand charges')
for xi, (tot, pct) in enumerate(zip(res['total_sav'], res['sav_pct'])):
    ax.text(xi, top * 1.05, f'${tot/1e3:,.0f}k ({pct:.1f}%)', ha='center', fontsize=9)
ax.set_ylim(min(0, res[['energy_sav', 'demand_sav']].min().min() / 1e3 * 1.1), top * 1.2)
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Annual savings ($ thousand)')
ax.set_title('Plot 5: annual savings by bill component (total and % of bill above bars)')
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

## 6. What if the plant can flex more, or less?

**What to look for**
- *Left (6a):* bill savings as a % of the annual bill as the flexed share grows. Under tariff B, shed saves a little (energy only), shift saves nothing (flat energy price) and peak shave saves the most because most of the bill is demand charge. Under tariff A, shed saves more than shift because shifted energy is bought again at the off-peak price.
- *Right (6b):* the reduction in the billed monthly peak. Shed and shift stay at 0% because the monthly peaks are outside the window. Peak shave equals the flexed share by construction (the cap is (1 - share) x the monthly max), so treat that line as a check, not a finding. It also assumes all clipped energy can be made up the same day, which a real plant may not manage.


In [ ]:
# Plot 6: sensitivity to the flexed share
base_billed = load.groupby(cal['month']).max().sum()
sweep = []
for f in np.arange(0, 0.31, 0.05):
    for sname, fn in (('Shed', shed), ('Shift', shift), ('Peak shave', shave)):
        kw = fn(load, f)
        row = dict(frac=100 * f, strategy=sname,
                   peak_red_pct=100 * (1 - kw.groupby(cal['month']).max().sum() / base_billed))
        for tname in TARIFFS:
            row[tname] = evaluate(kw, tname)['sav_pct']
        sweep.append(row)
sweep = pd.DataFrame(sweep)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
colors = dict(zip(TARIFFS, TARIFF_COLORS))
styles = {'Shed': '-', 'Shift': '--', 'Peak shave': ':'}
markers = {'Shed': 'o', 'Shift': 'x', 'Peak shave': 's'}
for tname in TARIFFS:
    for sname in styles:
        d = sweep[sweep['strategy'] == sname]
        ax1.plot(d['frac'], d[tname], styles[sname], color=colors[tname], marker='o', label=f'{sname}, tariff {tname[0]}')
ax1.set_xlabel('Share of load flexed (%)')
ax1.set_ylabel('Annual bill savings (%)')
ax1.set_title('Plot 6a: bill savings depend on the rate design')
ax1.legend(fontsize=8)
for sname in styles:
    d = sweep[sweep['strategy'] == sname]
    ax2.plot(d['frac'], d['peak_red_pct'], styles[sname], color='k', marker=markers[sname], ms=8, mfc='none', label=sname)
ax2.set_xlabel('Share of load flexed (%)')
ax2.set_ylabel('Reduction in billed monthly peak (%)')
ax2.set_title('Plot 6b: reduction in billed monthly peak (same under both tariffs)')
ax2.legend()
plt.tight_layout()
plt.show()

## Takeaways and discussion questions

1. **Peak demand is a property of the load.** It changes the same way under both tariffs, and only if the strategy touches the hour when the monthly peak happens (Plot 1b, Plot 4b).
2. **Rate design decides what is worth doing.** A TOU energy tariff pays for moving energy out of the peak window. A demand-charge tariff pays only for lowering the highest 15-minute reading (Plot 5, Plot 6a).
3. **Shed saves energy cost, shift does not.** Shift only changes *when* energy is used, so under a flat energy price it saves nothing on energy.

**Discuss**
- If the monthly peak happens at night, what could a plant do that shedding in the evening cannot?
- Which production steps could realistically shift by a few hours? What would that cost in output or quality?
- What would change if the demand charge used a ratchet (a minimum billed demand based on the prior year's peak)?

The tariffs, the scale factor and the flex percentages are assumptions. Treat the output as a teaching example, not a forecast for any real plant.